# ASSIGNMENT 1 - Question 10 - Multilingual Customer-Support Assistant
### Alumno: Martín Andrés Gutiérrez López
Topics NLP tokenization pipelines, dense token embeddings, bidirectional LSTMs, Encoder-Decoder (Seq2Seq) bottleneck

In [4]:
# ==============================================================================
# Question 10: Multilingual Customer-Support Assistant (Case: ShopAndes)
# Tokenization Pipeline, Embedding Layer Audit, BiLSTM Architecture & Parameters
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (a): Simulated Tokenization Pipeline Stages
# ------------------------------------------------------------------------------
raw_ticket = "The delivery wasn't on-time!!! Refund ASAP 😡 order #A-7781"

# 1. Normalization (lowercasing, punctuation normalization)
normalized_text = raw_ticket.lower()

# 2. Pre-tokenization (basic whitespace / symbol split)
pre_tokens = [
    "the",
    "delivery",
    "wasn't",
    "on-time",
    "!",
    "!",
    "!",
    "refund",
    "asap",
    "😡",
    "order",
    "#a-7781",
]

# 3. Subword modeling (splitting complex/unseen tokens: wasn't, ASAP, #A-7781)
subwords = [
    "the",
    "delivery",
    "was",
    "n't",
    "on",
    "-",
    "time",
    "!",
    "!",
    "!",
    "refund",
    "as",
    "##ap",
    "😡",
    "order",
    "#",
    "a",
    "-",
    "77",
    "##81",
]

# 4. Special tokens addition
special_token_sequence = ["[CLS]"] + subwords + ["[SEP]"]

# 5. ID mapping (sample vocabulary hash mapping)
vocab_sample = {
    token: 1000 + idx for idx, token in enumerate(special_token_sequence)
}
vocab_sample["[PAD]"] = 0
token_ids = [vocab_sample[t] for t in special_token_sequence]

# 6. Padding/truncation to fixed length (e.g., 25 tokens)
max_length = 25
if len(token_ids) < max_length:
  padded_ids = token_ids + [0] * (max_length - len(token_ids))
else:
  padded_ids = token_ids[:max_length]

print(f"\n{BOLD_BLUE}")
print("=" * 80)
print("PART (a): TOKENIZATION PIPELINE AUDIT")
print("=" * 80,f"{RESET}")
print(f"Raw Input Text         : {raw_ticket}")
print(f"Normalized Text        : {normalized_text}")
print(f"Subword Tokens (BPE)   : {subwords}")
print(f"With Special Tokens    : {special_token_sequence}")
print(f"Token IDs              : {token_ids[:6]} ... (length={len(token_ids)})")
print(f"Padded IDs (L={max_length})     : {padded_ids}")
print()

# ------------------------------------------------------------------------------
# Part (b): Embedding Layer Parameters & Representation Comparison
# ------------------------------------------------------------------------------
vocab_size = 30000
embed_dim = 300

embedding_layer = nn.Embedding(
    num_embeddings=vocab_size, embedding_dim=embed_dim
)
embedding_params = sum(p.numel() for p in embedding_layer.parameters())

print(f"{BOLD_BLUE}")
print("=" * 80)
print("PART (b): EMBEDDING LAYER PARAMETERS & DENSITY")
print("=" * 80,f"{RESET}")
print(f"Vocabulary Size (V)          : {vocab_size:,}")
print(f"Embedding Dimension (d)      : {embed_dim}")
print(
    f"Embedding Parameters (V x d) : {embedding_params:,} (9,000,000 parameters)"
)
print("Dense vs One-Hot Comparison:")
print("  - Dimensionality: Dense is 300-D continuous; One-hot is 30,000-D sparse.")
print(
    "  - Semantic Similarity: Dense captures cosine distance; One-hot vectors"
    " are orthogonal."
)
print()

# ------------------------------------------------------------------------------
# Part (c): BiLSTM Intent Classifier Output Shape & Parameters
# ------------------------------------------------------------------------------
hidden_dim_per_dir = 128
num_directions = 2

bilstm = nn.LSTM(
    input_size=embed_dim,
    hidden_size=hidden_dim_per_dir,
    batch_first=True,
    bidirectional=True,
    bias=True,
)

# Output shape per time step: 128 forward + 128 backward = 256
output_features_per_step = hidden_dim_per_dir * num_directions

# Analytical parameter formula (1 bias vector per gate):
# 4 gates * [(input_dim * hidden_dim) + (hidden_dim * hidden_dim) + hidden_dim] * 2 directions
params_per_gate_one_dir = (
    (embed_dim * hidden_dim_per_dir)
    + (hidden_dim_per_dir * hidden_dim_per_dir)
    + hidden_dim_per_dir
)
params_one_direction = 4 * params_per_gate_one_dir
total_bilstm_params_analytical = params_one_direction * num_directions

# PyTorch parameter count with single bias per gate adjustment
pytorch_single_bias_params = sum(
  layer.numel()
  for name, layer in bilstm.named_parameters()
  if "bias_hh" not in name
)

print(f"{BOLD_BLUE}")
print("=" * 80)
print("PART (c): BiLSTM CLASSIFIER SPECIFICATIONS")
print("=" * 80,f"{RESET}")
print(f"Output Feature Size per Step : {output_features_per_step} (128 forward + 128 backward)")
print(f"Parameters per direction     : {params_one_direction:,}")
print(f"Total BiLSTM Parameters      : {total_bilstm_params_analytical:,} (Both directions, 1 bias/gate)")
print(f"PyTorch Implementation Audit : {pytorch_single_bias_params:,}")

# Forward pass verification
dummy_input = torch.randint(0, vocab_size, (2, max_length))  # Batch=2, Seq=25
embedded = embedding_layer(dummy_input)  # Shape: (2, 25, 300)
lstm_out, (h_n, c_n) = bilstm(embedded)

print(f"Input Token Batch Shape      : {list(dummy_input.shape)}  (Batch, Sequence)")
print(f"Embedded Tensor Shape        : {list(embedded.shape)} (Batch, Sequence, Embed_Dim)")
print(f"BiLSTM Output Tensor Shape   : {list(lstm_out.shape)}  (Batch, Sequence, 256)")
print()

# ------------------------------------------------------------------------------
# Part (d): Information Bottleneck vs Attention Mechanism Simulation
# ------------------------------------------------------------------------------
print(f"{BOLD_BLUE}")
print("=" * 80)
print("PART (d): SEQ2SEQ BOTTLENECK & ATTENTION DYNAMICS")
print("=" * 80,f"{RESET}")
# Demonstration of Bahdanau Additive Attention weights computation
# Given encoder states (Batch=1, Seq=45, Hidden=256) and current decoder state (Batch=1, Hidden=256)
encoder_states = torch.randn(1, 45, 256)
decoder_state = torch.randn(1, 1, 256)

# Energy scores via dot product attention: (Batch, 1, 45)
attention_scores = torch.bmm(decoder_state, encoder_states.transpose(1, 2))
attention_weights = torch.softmax(attention_scores, dim=-1)

# Dynamic context vector: weighted sum of encoder representations
dynamic_context_vector = torch.bmm(attention_weights, encoder_states)

print(f"Encoder Sequence Length      : {encoder_states.shape[1]} tokens (> 40 threshold)")
print(f"Attention Weights Shape      : {list(attention_weights.shape)} (Distribution over all 45 inputs)")
print(f"Dynamic Context Vector Shape : {list(dynamic_context_vector.shape)} (Replaces single static vector)")
print("Result: Attention resolves the fixed-length compression bottleneck.")
   



PART (a): TOKENIZATION PIPELINE AUDIT
Raw Input Text         : The delivery wasn't on-time!!! Refund ASAP 😡 order #A-7781
Normalized Text        : the delivery wasn't on-time!!! refund asap 😡 order #a-7781
Subword Tokens (BPE)   : ['the', 'delivery', 'was', "n't", 'on', '-', 'time', '!', '!', '!', 'refund', 'as', '##ap', '😡', 'order', '#', 'a', '-', '77', '##81']
With Special Tokens    : ['[CLS]', 'the', 'delivery', 'was', "n't", 'on', '-', 'time', '!', '!', '!', 'refund', 'as', '##ap', '😡', 'order', '#', 'a', '-', '77', '##81', '[SEP]']
Token IDs              : [1000, 1001, 1002, 1003, 1004, 1005] ... (length=22)
Padded IDs (L=25)     : [1000, 1001, 1002, 1003, 1004, 1005, 1018, 1007, 1010, 1010, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 0, 0, 0]


PART (b): EMBEDDING LAYER PARAMETERS & DENSITY
Vocabulary Size (V)          : 30,000
Embedding Dimension (d)      : 300
Embedding Parameters (V x d) : 9,000,000 (9,000,000 parameters)
Dense vs One-Hot Compari